# 04 — Memory Bank: Coreset Subsampling

**Author:** Sherif Abd El-Rady

**Objective:** Build the PatchCore memory bank from training-image patch embeddings using greedy coreset subsampling. Analyse how the bank size affects coverage and inference speed.

## Why not store all patches?

With 209 training images × 28×28 feature map = **163,904 patch vectors** at 1536-D each, storing all patches is 950 MB and k-NN over them takes seconds per image. Coreset subsampling keeps a geometrically representative 1% (~1,639 vectors) with negligible coverage loss.

## Greedy coreset selection

```
1. Start with a random seed point s₀
2. For i = 1 to K:
   a. For each unselected point p, compute dist(p, nearest selected)
   b. Select the point with maximum distance (farthest from existing set)
3. Result: K points that maximally cover the data manifold
```

This is a greedy approximation of the k-centre problem. It minimises the maximum distance from any data point to its nearest selected centre.


In [ ]:
import sys, time
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

from defectdetect import FeatureExtractor, MemoryBank

ROOT     = Path('..').resolve()
MVTEC    = ROOT / 'data' / 'mvtec'
CATEGORY = 'bottle'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

if not (MVTEC / CATEGORY / 'train' / 'good').exists():
    MVTEC = ROOT / 'data' / 'synthetic'; CATEGORY = ''
    print('Using synthetic dataset')

base      = MVTEC / CATEGORY if CATEGORY else MVTEC
train_dir = base / 'train' / 'good'
BANK_DIR  = ROOT / 'reports' / 'memory_banks'
BANK_DIR.mkdir(parents=True, exist_ok=True)
print('Ready')

In [ ]:
# ── 4.1  Extract all training patches ─────────────────────────────────────────
extractor    = FeatureExtractor(layers=['layer2','layer3'], input_size=224, device=DEVICE)
train_paths  = sorted(train_dir.glob('*.png')) + sorted(train_dir.glob('*.jpg'))
BATCH        = 8
all_patches  = []

for i in tqdm(range(0, len(train_paths), BATCH), desc='Extracting'):
    imgs   = [cv2.imread(str(p)) for p in train_paths[i:i+BATCH] if cv2.imread(str(p)) is not None]
    if not imgs: continue
    batch  = extractor.preprocess(imgs)
    pat, _ = extractor.extract_patches(batch)
    all_patches.append(pat)

train_patches = np.vstack(all_patches)
print(f'Total patches: {train_patches.shape[0]:,}  dim={train_patches.shape[1]}')

In [ ]:
# ── 4.2  Coreset at multiple ratios — analyse coverage vs size ─────────────────
ratios = [0.001, 0.005, 0.01, 0.05, 0.1]
coverage_stats = []

for ratio in ratios:
    bank = MemoryBank(sampling_ratio=ratio, projection_dim=128)
    t0   = time.perf_counter()
    bank.fit(train_patches)
    fit_ms = (time.perf_counter() - t0) * 1000

    # Coverage: mean distance from each training patch to its nearest bank entry
    sample_idx = np.random.choice(len(train_patches), min(500, len(train_patches)), replace=False)
    scores = bank.score(train_patches[sample_idx], k=1)
    coverage_stats.append({
        'ratio': ratio,
        'n_selected': bank.size,
        'mean_dist': scores.mean(),
        'max_dist': scores.max(),
        'fit_ms': fit_ms,
    })
    print(f'ratio={ratio:.3f}  bank={bank.size:5d}  mean_dist={scores.mean():.4f}  fit={fit_ms:.0f}ms')

In [ ]:
# ── 4.3  Visualise coverage vs bank size trade-off ────────────────────────────
sizes   = [s['n_selected'] for s in coverage_stats]
dists   = [s['mean_dist']  for s in coverage_stats]
ratios_ = [s['ratio']      for s in coverage_stats]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(sizes, dists, 'o-', color='#3498db', linewidth=2, markersize=8)
for s, d, r in zip(sizes, dists, ratios_):
    axes[0].annotate(f'{r:.1%}', (s, d), textcoords='offset points', xytext=(5, 5), fontsize=8)
axes[0].set_xlabel('Memory bank size (number of patches)')
axes[0].set_ylabel('Mean distance to nearest bank entry')
axes[0].set_title('Coverage vs Bank Size\n(lower distance = better coverage)')
axes[0].grid(True, alpha=0.3)

fit_times = [s['fit_ms'] for s in coverage_stats]
axes[1].bar([str(r) for r in ratios_], fit_times, color='#e74c3c', alpha=0.8)
axes[1].set_xlabel('Sampling ratio')
axes[1].set_ylabel('Fit time (ms)')
axes[1].set_title('Coreset construction time vs ratio')

plt.suptitle('Memory Bank — Coreset Subsampling Analysis', fontsize=11)
plt.tight_layout()
plt.show()

print('Recommendation: 0.01 (1%) provides ~90% of the coverage of 10% with 10× smaller bank.')

In [ ]:
# ── 4.4  Build and save the production memory bank ────────────────────────────
SAMPLING_RATIO = 0.01

bank = MemoryBank(sampling_ratio=SAMPLING_RATIO, projection_dim=128)
bank.fit(train_patches)

bank_path = BANK_DIR / f'{CATEGORY or "synthetic"}_bank_r{SAMPLING_RATIO}.pkl'
bank.save(bank_path)

# Verify round-trip
loaded = MemoryBank.load(bank_path)
assert loaded.size == bank.size, 'Save/load size mismatch!'
print(f'✓ Memory bank saved and verified: {bank.size} entries → {bank_path}')
print(f'  Sampling ratio: {SAMPLING_RATIO} ({SAMPLING_RATIO:.0%} of {len(train_patches):,} patches)')

## Summary

| Metric | Value |
|---|---|
| Total training patches | *(see Cell 4.1)* |
| Memory bank size (1%) | *(see Cell 4.4)* |
| Projection dim | 128 (from 1536-D) |
| Bank file | `reports/memory_banks/*.pkl` |

## Next step

Open `05_patchcore_inference.ipynb` to score test images using the memory bank.
